# S001 + Vol Control — LSE Yield + Dividend Growth

## Strategy Overview

**Economic rationale:** Dividend-paying stocks on the LSE tend to be mature, cash-generative businesses. Ranking on *both* yield and dividend growth identifies companies that pay well today **and** are growing their payout — a proxy for management confidence in future earnings. The vol-control overlay protects capital during market stress by exiting to cash before the deepest drawdown days, exploiting the fact that realised vol leads price.

---

## Signal Construction

| Component | Weight | Method | Smoothing |
|-----------|--------|--------|-----------|
| Trailing-12M yield rank | **50%** | TTM dividends (365d rolling) ÷ close price, clipped at 15% | EWMA span=90d before ranking |
| YoY dividend growth rank | **50%** | (TTM_now ÷ TTM_prev_year) − 1, clipped [−100%, +500%] | EWMA span=60d before ranking |

Both components ranked cross-sectionally each month (percentile 0–1). Composite = 0.5 × yield_rank + 0.5 × growth_rank (equal weighting chosen by sweep as best Sharpe).  
EWMA smoothing reduces noise from lumpy semi-annual/annual payers without introducing look-ahead bias (applied to per-ticker time series, not cross-sectionally).

---

## Portfolio & Vol Gate

| Parameter | Value |
|-----------|-------|
| Universe | LSE Common Stocks only — ~3,997 tickers, ETFs excluded via `eodhd.exchange_tickers WHERE type='Common Stock'` |
| Selection | Long-only Top-10 by composite score, equal-weight |
| Rebalance | Monthly (end-of-month) |
| Execution | **Market on Close (MOC)** — signal on day T executes at T+1 close (`signal_delay_bars=1`) |
| Vol gate | **All positions closed to cash** when 252d z-score of 21d realised vol > 2.5. Median computed across all ~3,997 investable Common Stocks (no ETFs). `regime_ok` broadcasts to all instruments; when False, `TopN` selects 0 names → full liquidation. ⚠️ Gate fires at **month-end rebalance only** — mid-month vol spikes do not trigger intra-month exit. |
| Costs | 10 bps commission per trade (one-way) + power-law slippage (1 bps base, k=5) |
| Outputs | `outputs/vol_control/` |

**Why median vol, not mean?** A single bad-data tick (e.g. `3SVP`: +2,334,900% in one day) dragged the cross-sectional *mean* from ~13% to ~8,456%, corrupting the z-score window for a full year. Median of ~3,997 tickers is immune to any number of outliers.


In [1]:
import sys, os, warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path(r"c:\Personal\Business & Investments\Python codes")))
from signum import Chart
from signum.engine.dashboard import Dashboard
from signum.engine.statchart import StatChart

def _find_btest_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "AGENT_DSL_REFERENCE.md").exists():
            return p
    return Path(r"c:\Personal\Business & Investments\Python codes\btest")

BTEST_ROOT  = _find_btest_root()
os.chdir(BTEST_ROOT)

_btest_src = str(BTEST_ROOT / "src")
if _btest_src not in sys.path:
    sys.path.insert(0, _btest_src)

SIGNAL_ROOT = Path("research/generated/Dividend Growth/signals/001_yield_growth_composite")
OUTPUTS     = SIGNAL_ROOT / "outputs" / "vol_control"   # ← vol-control strategy outputs
DATA_DIR    = SIGNAL_ROOT / "data"
SHARED_DATA = Path("research/generated/Dividend Growth/shared_data")

equity  = pd.read_parquet(OUTPUTS / "equity.parquet")
returns = pd.read_parquet(OUTPUTS / "returns.parquet")
trades  = pd.read_parquet(OUTPUTS / "trades.parquet")
weights = pd.read_parquet(OUTPUTS / "weights.parquet")

raw_sum = json.loads((OUTPUTS / "summary.json").read_text())
summary = raw_sum.get("metrics", raw_sum)

for df_ in [equity, returns, weights]:
    if hasattr(df_.index, "tz") and df_.index.tz is not None:
        df_.index = df_.index.tz_localize(None)

eq        = equity.iloc[:, 0].dropna()
strat_ret = returns.iloc[:, 0].dropna()

print(f"Loaded vol_control outputs: {eq.index[0].date()} → {eq.index[-1].date()}")
print(f"Total return: {eq.iloc[-1]/eq.iloc[0]-1:.1%}")


Loaded vol_control outputs: 2015-01-02 → 2025-12-31
Total return: 368.2%


## Performance Summary

In [2]:
ann = 252
r    = strat_ret.dropna()
cagr = (eq.iloc[-1] / eq.iloc[0]) ** (ann / len(r)) - 1
dd   = (eq - eq.cummax()) / eq.cummax()

metrics = {
    "Total Return"    : f"{eq.iloc[-1]/eq.iloc[0]-1:.1%}",
    "CAGR"            : f"{cagr:.1%}",
    "Sharpe"          : f"{r.mean()/r.std()*np.sqrt(ann):.2f}",
    "Sortino"         : f"{r.mean()/r[r<0].std()*np.sqrt(ann):.2f}",
    "Max Drawdown"    : f"{dd.min():.1%}",
    "Calmar"          : f"{cagr/abs(dd.min()):.2f}",
    "Ann Volatility"  : f"{r.std()*np.sqrt(ann):.1%}",
    "Avg Daily Trades": f"{len(trades)/len(eq):.1f}",
}

mdf = pd.DataFrame.from_dict(metrics, orient="index", columns=["Value"])
display(mdf.style
    .set_caption("S001 — LSE Yield + Dividend Growth  |  Key Metrics")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size","14px"),("font-weight","bold"),("text-align","left")]},
        {"selector": "th",      "props": [("text-align","left")]},
        {"selector": "td",      "props": [("text-align","right"),("font-family","monospace")]},
    ])
    .set_properties(**{"width": "140px"})
)


,Value
Total Return,368.2%
CAGR,15.0%
Sharpe,1.12
Sortino,1.44
Max Drawdown,-22.9%
Calmar,0.66
Ann Volatility,13.3%
Avg Daily Trades,0.5


## Equity Curve & Drawdown

In [3]:

nav_idx  = eq / eq.iloc[0] * 100
dd_pct   = (eq - eq.cummax()) / eq.cummax() * 100
r_sharpe = (strat_ret.rolling(63, min_periods=63).mean()
            / strat_ret.rolling(63, min_periods=63).std()
            * np.sqrt(252))

nav_df = pd.DataFrame({"time": nav_idx.index, "value": nav_idx.values})
dd_df  = pd.DataFrame({"time": dd_pct.index,  "value": dd_pct.values})
rs_df  = pd.DataFrame({"time": r_sharpe.index, "value": r_sharpe.values})

ann = 252
r    = strat_ret.dropna()
cagr = (eq.iloc[-1] / eq.iloc[0]) ** (ann / len(r)) - 1
dd_min = dd_pct.min()
sharpe_val = r.mean() / r.std() * np.sqrt(ann)
sortino_val = r.mean() / r[r < 0].std() * np.sqrt(ann)
calmar_val  = cagr / abs(dd_min / 100)

# Signal coverage stats
composite = pd.read_parquet(DATA_DIR / "div_composite.parquet")
if hasattr(composite.index, "tz") and composite.index.tz is not None:
    composite.index = composite.index.tz_localize(None)
n_scored = composite.notna().sum(axis=1).resample("ME").mean()
n_held   = (weights > 0.001).sum(axis=1).resample("ME").mean()

sc_df  = pd.DataFrame({"time": n_scored.index, "value": n_scored.values})
hld_df = pd.DataFrame({"time": n_held.index,   "value": n_held.values})

nav_chart = (
    Chart(height=260)
    .area(nav_df, name="NAV (rebased 100)", color="#26a69a")
    .stats_legend({
        "Total Return": f"{eq.iloc[-1]/eq.iloc[0]-1:.1%}",
        "CAGR":         f"{cagr:.1%}",
        "Sharpe":       f"{sharpe_val:.2f}",
        "Sortino":      f"{sortino_val:.2f}",
        "Calmar":       f"{calmar_val:.2f}",
        "Max DD":       f"{dd_min:.1f}%",
        "Ann Vol":      f"{r.std()*np.sqrt(ann):.1%}",
    }, position="top-left")
)

Dashboard(
    panes=[
        nav_chart,
        Chart(height=110).area(dd_df,   name="Drawdown %",        color="#ef5350"),
        Chart(height=110).baseline(rs_df, base_value=0, value_col="value"),
        Chart(height=100).area(sc_df,   name="Tickers scored",    color="#1976d2"),
        Chart(height=100).area(hld_df,  name="Positions held",    color="#ff9800"),
    ],
    titles=[
        f"NAV  ·  CAGR {cagr:.1%}  ·  Sharpe {sharpe_val:.2f}  ·  Total Return {eq.iloc[-1]/eq.iloc[0]-1:.1%}",
        f"Drawdown  ·  Max {dd_min:.1f}%",
        f"Rolling Sharpe (63d)  ·  Full-period {sharpe_val:.2f}",
        f"Signal Coverage (div payers)  ·  avg {n_scored.mean():.0f} tickers/month",
        f"Positions Held  ·  avg {n_held.mean():.0f}",
    ],
    theme="dark",
)


## Monthly Returns Heatmap

In [4]:
monthly = strat_ret.resample("ME").apply(lambda x: (1+x).prod()-1)
pivot = monthly.rename_axis("date").to_frame("ret")
pivot["year"] = pivot.index.year; pivot["month"] = pivot.index.month
pivot = pivot.pivot(index="year", columns="month", values="ret") * 100
pivot.columns = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
pivot["Annual"] = strat_ret.resample("YE").apply(lambda x: (1+x).prod()-1) * 100

display(
    pivot.style
    .format("{:.1f}%", na_rep="")
    .background_gradient(cmap="RdYlGn", vmin=-8, vmax=8, subset=list(pivot.columns[:-1]))
    .background_gradient(cmap="RdYlGn", vmin=-20, vmax=20, subset=["Annual"])
    .set_caption("S001 — Monthly Returns (%)")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size","13px"),("font-weight","bold")]},
        {"selector": "th",      "props": [("text-align","center"),("min-width","48px")]},
        {"selector": "td",      "props": [("text-align","right"), ("font-family","monospace"),("min-width","48px")]},
    ])
)

# Annual returns distribution
ann_ret = strat_ret.resample("YE").apply(lambda x: (1+x).prod()-1)
StatChart(theme="dark", height=220, title="Annual Returns Distribution").distribution(
    ann_ret * 100, bins=14, name="Annual Return %", color="#26a69a",
    show_mean=True, show_median=True,
).show()


,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,Annual
year,,,,,,,,,,,,,
2015,0.0%,0.0%,0.0%,0.0%,0.0%,0.1%,0.6%,-0.7%,-2.0%,3.4%,1.8%,-1.6%,
2016,-6.4%,2.2%,-0.9%,-0.2%,3.8%,-8.1%,4.7%,5.7%,-1.1%,0.8%,1.3%,5.3%,
2017,0.5%,1.6%,6.3%,2.9%,4.3%,0.9%,7.2%,0.0%,3.1%,4.8%,-4.3%,7.0%,
2018,-0.2%,-1.6%,-1.8%,6.5%,3.9%,1.0%,4.6%,2.4%,1.1%,-7.2%,1.7%,-3.4%,
2019,4.7%,5.7%,1.2%,7.3%,-2.7%,5.6%,1.6%,-4.1%,2.4%,-4.2%,4.3%,7.2%,
2020,-1.3%,-7.6%,0.4%,0.0%,6.9%,3.4%,-0.5%,6.0%,-3.9%,0.2%,17.1%,6.1%,
2021,-1.8%,1.3%,3.7%,2.9%,1.0%,0.2%,0.8%,1.8%,-0.7%,1.5%,0.2%,4.4%,
2022,-0.6%,-4.3%,-2.0%,-0.1%,-1.1%,-5.2%,7.2%,-10.1%,-5.7%,4.3%,8.1%,-0.1%,
2023,5.8%,2.8%,-2.2%,4.3%,0.1%,1.2%,7.0%,-1.4%,3.5%,-6.3%,4.0%,4.2%,


## Portfolio Holdings Drilldown

In [5]:
latest_w = weights.iloc[-1].dropna()
latest_w = latest_w[latest_w > 0.001].sort_values(ascending=False) * 100

wdf = latest_w.reset_index()
wdf.columns = ["Ticker", "Weight %"]

# --- Look up names + type from eodhd reference data ---
import psycopg
sys.path.insert(0, r"c:\Personal\Business & Investments\Python codes\sfera")
from data.config.database_config import DB_CONFIG

_tickers = wdf["Ticker"].tolist()
with psycopg.connect(**DB_CONFIG) as _conn:
    with _conn.cursor() as _cur:
        _cur.execute(
            "SELECT ticker, name, type FROM eodhd.exchange_tickers "
            "WHERE ticker = ANY(%s) AND exchange = 'LSE'",
            [_tickers]
        )
        _rows = _cur.fetchall()

df_names = pd.DataFrame(_rows, columns=["Ticker", "Name", "Type"])
wdf = wdf.merge(df_names, on="Ticker", how="left")
wdf.index = range(1, len(wdf) + 1)

# Flag non-Common-Stock rows
_is_equity = wdf["Type"] == "Common Stock"
print(f"Holdings classified as Common Stock: {_is_equity.sum()}/{len(wdf)}")
_non_eq = wdf[~_is_equity]
if len(_non_eq):
    print(f"⚠️  Non-Common-Stock in holdings:\n{_non_eq[['Ticker','Name','Type']].to_string()}")

display(
    wdf.style
    .format({"Weight %": "{:.2f}%"})
    .bar(subset=["Weight %"], color="#26a69a", vmin=0)
    .apply(lambda row: ["background-color: #3a2000" if row["Type"] != "Common Stock" else "" for _ in row], axis=1)
    .set_caption(f"S001 — Current Holdings at {weights.index[-1].date()}  ({len(wdf)} positions)")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size","13px"),("font-weight","bold")]},
        {"selector": "td.col0", "props": [("font-weight","bold"),("font-family","monospace")]},
    ])
)

# Weight over time for top-5 positions
top5 = latest_w.head(5).index.tolist()
w_top5 = weights[top5].fillna(0) * 100
w_top5.index = pd.to_datetime(w_top5.index).normalize()
w_top5_df = w_top5.reset_index()
w_top5_df.columns = ["time"] + top5

chart = Chart(height=220, theme="dark", watermark="Top-5 Weights over Time")
for tk in top5:
    chart.line(w_top5_df[["time", tk]].rename(columns={tk: "value"}), name=tk)
chart


Holdings classified as Common Stock: 10/10


,Ticker,Weight %,Name,Type
1,BGEO,10.50%,Lion Finance Group PLC,Common Stock
2,JCH,10.33%,JPMorgan Claverhouse Investment Trust Plc,Common Stock
3,DNLM,10.24%,Dunelm Group PLC,Common Stock
4,MNL,10.22%,Manchester and London Investment Trust plc,Common Stock
5,VTAS,10.22%,Volta Finance Ltd,Common Stock
6,TBCG,10.01%,TBC Bank Group PLC,Common Stock
7,FOUR,9.92%,4Imprint Group Plc,Common Stock
8,HFG,9.91%,Hilton Food Group Plc,Common Stock
9,GAW,9.73%,Games Workshop Group PLC,Common Stock
10,TFGS,8.94%,Tetragon Financial Group Ltd,Common Stock


## Signal Coverage Over Time

In [6]:
composite = pd.read_parquet(DATA_DIR / "div_composite.parquet")
if hasattr(composite.index, "tz") and composite.index.tz is not None:
    composite.index = composite.index.tz_localize(None)

n_scored  = composite.notna().sum(axis=1).resample("ME").mean()
n_held    = (weights > 0.001).sum(axis=1).resample("ME").mean()

# Align on common months
common_idx = n_scored.index.intersection(n_held.index)
n_scored   = n_scored.loc[common_idx]
n_held     = n_held.loc[common_idx]

cov_df = pd.DataFrame({
    "time"    : n_scored.index,
    "scored"  : n_scored.values,
    "held"    : n_held.values,
})

Dashboard(
    panes=[
        Chart(height=180).area(cov_df[["time","scored"]].rename(columns={"scored":"value"}),
                               name="Tickers scored", color="#1976d2"),
        Chart(height=130).area(cov_df[["time","held"]].rename(columns={"held":"value"}),
                               name="Positions held", color="#ff9800"),
    ],
    titles=[
        f"Signal Coverage (dividend payers)  ·  avg {n_scored.mean():.0f} tickers/month",
        f"Avg Positions Held  ·  avg {n_held.mean():.0f}",
    ],
    theme="dark",
)


## Trade Analysis — Round-Trip P&L

FIFO-matched closed round-trips per instrument. Shows entry/exit dates, holding period, cost basis (amount invested), proceeds (amount received), and gross P&L before commissions.


In [7]:
from quantdsl_backtest.utils.trades import build_roundtrips, roundtrip_summary

rt  = build_roundtrips(trades)
smy = roundtrip_summary(rt)

print(f"Round-trips (closed): {smy['n_trips']:,}  |  "
      f"Win rate: {smy['win_rate']:.1%}  |  "
      f"Profit factor: {smy['profit_factor']:.2f}")
print(f"Avg hold: {smy['avg_hold_days']:.0f}d  |  Median hold: {smy['median_hold_days']:.0f}d")
print(f"Avg win (net): £{smy['avg_win']:,.0f}  |  Avg loss (net): £{smy['avg_loss']:,.0f}")
print(f"Total gross P&L: £{smy['total_gross_pnl']:,.0f}  |  "
      f"Total commission drag: £{smy['total_commission']:,.0f}  |  "
      f"Total net P&L: £{smy['total_net_pnl']:,.0f}")
print(f"Note: engine uses fractional shares (no round-lot rounding). "
      f"Commission baked in per trade; slippage baked into execution price.")

_fmt = {
    "cost_basis"       : "£{:,.0f}",
    "proceeds"         : "£{:,.0f}",
    "gross_pnl"        : "£{:,.0f}",
    "gross_pnl_pct"    : "{:.1f}%",
    "total_commission" : "£{:,.0f}",
    "net_pnl"          : "£{:,.0f}",
    "net_pnl_pct"      : "{:.1f}%",
}

# Holding period bucket table
hold_bins = pd.cut(rt["holding_days"],
                   bins=[0, 7, 14, 30, 60, 90, 180, 365, 9999],
                   labels=["≤7d","8–14d","15–30d","31–60d","61–90d","91–180d","181d–1y",">1y"])
hold_dist = rt.groupby(hold_bins, observed=True).agg(
    count=("net_pnl","count"),
    avg_net_pnl=("net_pnl","mean"),
    win_rate=("net_pnl", lambda x: (x > 0).mean() * 100),
    avg_commission=("total_commission","mean"),
).reset_index()
hold_dist.columns = ["Holding bucket","Count","Avg net P&L (£)","Win rate %","Avg commission (£)"]
display(hold_dist.style
    .format({"Avg net P&L (£)": "£{:,.0f}", "Win rate %": "{:.0f}%", "Avg commission (£)": "£{:,.0f}"})
    .background_gradient(subset=["Win rate %"], cmap="RdYlGn", vmin=30, vmax=70)
    .set_caption("Round-Trips by Holding Period"))

# Scatter: holding days vs net P&L %
StatChart(theme="dark", height=340,
          title="Holding Days vs Net P&L %  (each dot = one round-trip, after commission)").scatter(
    rt["holding_days"].values,
    rt["net_pnl_pct"].values,
    name="Round-trip",
    color="#26a69a",
    size=3,
).show()


Round-trips (closed): 1,180  |  Win rate: 70.8%  |  Profit factor: 2.80
Avg hold: 173d  |  Median hold: 122d
Avg win (net): £6,274  |  Avg loss (net): £-5,414
Total gross P&L: £3,491,325  |  Total commission drag: £120,382  |  Total net P&L: £3,370,943
Note: engine uses fractional shares (no round-lot rounding). Commission baked in per trade; slippage baked into execution price.


,Holding bucket,Count,Avg net P&L (£),Win rate %,Avg commission (£)
0,15–30d,111,£-63,54%,£163
1,31–60d,175,£602,60%,£143
2,61–90d,135,£950,74%,£119
3,91–180d,319,"£2,023",66%,£90
4,181d–1y,311,"£3,871",77%,£80
5,>1y,129,"£10,040",94%,£58


---

## Per-Ticker Attribution

Decompose portfolio returns by individual ticker using daily `weight × price return`. Shows which names drove performance, how long each was held, and trade-level realized P&L.

In [8]:
# === Per-Ticker Return Attribution ===
prices_long = pd.read_parquet(SHARED_DATA / "lse_prices.parquet")
prices_wide = prices_long.pivot_table(index="date", columns="ticker", values="close")
price_ret   = prices_wide.sort_index().pct_change(fill_method=None).clip(-0.5, 0.5)

w = weights.fillna(0.0)
w.index = pd.to_datetime(w.index).normalize()
price_ret.index = pd.to_datetime(price_ret.index).normalize()

# Diagnostics
print(f"weights tickers (sample): {w.columns[:5].tolist()}")
print(f"prices  tickers (sample): {price_ret.columns[:5].tolist()}")
print(f"weights dates:  {w.index[0].date()} → {w.index[-1].date()}  ({len(w)} rows)")
print(f"prices  dates:  {price_ret.index[0].date()} → {price_ret.index[-1].date()}  ({len(price_ret)} rows)")

common_tickers = w.columns.intersection(price_ret.columns)
common_dates   = w.index.intersection(price_ret.index)
print(f"common tickers: {len(common_tickers)}  |  common dates: {len(common_dates)}")

if len(common_tickers) == 0:
    raise ValueError(
        "No ticker overlap between weights and prices. "
        f"Weight sample: {w.columns[:3].tolist()}  "
        f"Price sample: {price_ret.columns[:3].tolist()}"
    )

w_sub = w.loc[common_dates, common_tickers]
r_sub = price_ret.loc[common_dates, common_tickers].fillna(0.0)

daily_contrib = w_sub.shift(1).fillna(0.0) * r_sub
ever_held     = (w_sub > 0.001).any()
contrib       = daily_contrib.loc[:, ever_held].sum()
n_days_held   = (w_sub.loc[:, ever_held] > 0.001).sum()
avg_wt        = w_sub.loc[:, ever_held].where(w_sub.loc[:, ever_held] > 0.001).mean()

attr = pd.DataFrame({
    "contrib_bps"    : (contrib * 10000).round(1),
    "contrib_pct"    : (contrib * 100).round(2),
    "days_held"      : n_days_held,
    "pct_time_held"  : (n_days_held / len(w_sub) * 100).round(1),
    "avg_weight_pct" : (avg_wt * 100).round(2),
}).sort_values("contrib_bps", ascending=False)

print(f"Tickers held: {ever_held.sum()} | Attribution sum: {contrib.sum()*100:.2f}% | Portfolio total: {(eq.iloc[-1]/eq.iloc[0]-1)*100:.2f}%")

top15 = attr.head(15).copy()
bot10 = attr.tail(10).copy()

display(top15.style
    .background_gradient(subset=["contrib_bps"], cmap="Greens")
    .format({"contrib_pct": "{:.2f}%", "pct_time_held": "{:.1f}%", "avg_weight_pct": "{:.2f}%"})
    .set_caption("Top 15 Contributors")
)
display(bot10.style
    .background_gradient(subset=["contrib_bps"], cmap="Reds_r")
    .format({"contrib_pct": "{:.2f}%", "pct_time_held": "{:.1f}%", "avg_weight_pct": "{:.2f}%"})
    .set_caption("Bottom 10 Detractors")
)


weights tickers (sample): ['0QKI', '0QO7', '0QP2', '0QPY', '3IN']
prices  tickers (sample): ['0A06', '0A07', '0A0A', '0A24', '0A3D']
weights dates:  2015-01-02 → 2025-12-31  (2779 rows)
prices  dates:  2013-01-02 → 2026-05-06  (3397 rows)
common tickers: 119  |  common dates: 2779
Tickers held: 68 | Attribution sum: 166.07% | Portfolio total: 368.15%


,contrib_bps,contrib_pct,days_held,pct_time_held,avg_weight_pct
GAW,1814.900000,18.15%,950,34.2%,10.06%
HSBA,1284.600000,12.85%,1345,48.4%,9.94%
TBCG,1115.500000,11.16%,778,28.0%,9.73%
BTRW,1031.200000,10.31%,1390,50.0%,9.78%
BGEO,1009.900000,10.10%,720,25.9%,9.91%
FOUR,966.300000,9.66%,761,27.4%,9.99%
BME,826.100000,8.26%,948,34.1%,9.90%
VTAS,703.500000,7.04%,736,26.5%,9.78%
ATY,574.300000,5.74%,442,15.9%,10.01%
JCH,548.900000,5.49%,653,23.5%,10.01%


,contrib_bps,contrib_pct,days_held,pct_time_held,avg_weight_pct
MAJE,-39.200000,-0.39%,86,3.1%,8.67%
FEV,-64.900000,-0.65%,63,2.3%,10.01%
SWR,-80.000000,-0.80%,404,14.5%,9.87%
GFTU,-95.300000,-0.95%,43,1.5%,9.73%
CCH,-127.000000,-1.27%,275,9.9%,9.89%
WPP,-144.200000,-1.44%,486,17.5%,9.96%
TFGS,-158.300000,-1.58%,277,10.0%,9.56%
QQ,-216.100000,-2.16%,43,1.5%,9.41%
CMCL,-257.100000,-2.57%,363,13.1%,7.85%
RHIM,-320.000000,-3.20%,632,22.7%,9.77%


In [9]:
# Attribution: cumulative contribution line for top-10 + bottom-5
# Each line = cumsum(daily weight × price return) for that ticker → bps added to portfolio over time
top10_tickers = attr.head(10).index.tolist()
bot5_tickers  = attr.tail(5).index.tolist()

cum_contrib_top = daily_contrib[top10_tickers].cumsum() * 10000
cum_contrib_bot = daily_contrib[bot5_tickers].cumsum() * 10000
cum_contrib_top.index = pd.to_datetime(cum_contrib_top.index).normalize()
cum_contrib_bot.index = pd.to_datetime(cum_contrib_bot.index).normalize()

chart_top = Chart(height=240, theme="dark")
for tk in top10_tickers:
    df_tk = cum_contrib_top[[tk]].reset_index()
    df_tk.columns = ["time", "value"]
    chart_top.line(df_tk, name=tk)

chart_bot = Chart(height=200, theme="dark")
for tk in bot5_tickers:
    df_tk = cum_contrib_bot[[tk]].reset_index()
    df_tk.columns = ["time", "value"]
    chart_bot.line(df_tk, name=tk)

Dashboard(
    panes=[chart_top, chart_bot],
    titles=[
        f"Top-10 Contributors — Cumulative return contribution (bps)  ·  total {attr.head(10)['contrib_bps'].sum():.0f} bps",
        f"Bottom-5 Detractors — Cumulative return contribution (bps)  ·  total {attr.tail(5)['contrib_bps'].sum():.0f} bps",
    ],
    theme="dark",
)


In [10]:

# Days held vs contribution scatter — winners green, losers red, labeled extremes
scatter_df = attr[attr["days_held"] > 0].copy()

winners = scatter_df[scatter_df["contrib_bps"] >= 0]
losers  = scatter_df[scatter_df["contrib_bps"] <  0]

sc = StatChart(theme="dark", height=400, title="Days Held vs Cumulative Contribution (bps)  ·  green = positive, red = negative")
if len(winners):
    sc.scatter(winners["days_held"].values, winners["contrib_bps"].values,
               name="Positive contributor", color="#26a69a", size=5)
if len(losers):
    sc.scatter(losers["days_held"].values, losers["contrib_bps"].values,
               name="Detractor",           color="#ef5350", size=5)
sc.show()

# Labeled extremes table (top-5 winners + bottom-5 losers)
top5_s  = scatter_df.nlargest(5,  "contrib_bps")[["contrib_bps","days_held","contrib_pct","avg_weight_pct"]]
bot5_s  = scatter_df.nsmallest(5, "contrib_bps")[["contrib_bps","days_held","contrib_pct","avg_weight_pct"]]
labeled = pd.concat([top5_s, bot5_s])
labeled.index.name = "Ticker"

display(labeled.style
    .format({"contrib_bps": "{:+.0f}", "contrib_pct": "{:+.2f}%",
             "days_held": "{:.0f}", "avg_weight_pct": "{:.2f}%"})
    .background_gradient(subset=["contrib_bps"], cmap="RdYlGn",
                         vmin=labeled["contrib_bps"].min(), vmax=labeled["contrib_bps"].max())
    .set_caption("Top-5 contributors & Bottom-5 detractors  (days held → bps)")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size","13px"),("font-weight","bold")]},
        {"selector": "th",      "props": [("text-align","left")]},
        {"selector": "td",      "props": [("text-align","right"),("font-family","monospace")]},
    ])
)


,contrib_bps,days_held,contrib_pct,avg_weight_pct
Ticker,,,,
GAW,+1815,950,+18.15%,10.06%
HSBA,+1285,1345,+12.85%,9.94%
TBCG,+1116,778,+11.16%,9.73%
BTRW,+1031,1390,+10.31%,9.78%
BGEO,+1010,720,+10.10%,9.91%
RHIM,-320,632,-3.20%,9.77%
CMCL,-257,363,-2.57%,7.85%
QQ,-216,43,-2.16%,9.41%
TFGS,-158,277,-1.58%,9.56%
